In [22]:
import math
import pandas as pd
from collections import defaultdict, deque


"""RAW
---------------------
"""


bom_df = pd.DataFrame([
    {"parent": "A", "component": "B", "qty_per": 2},
    {"parent": "A", "component": "C", "qty_per": 1},
    {"parent": "B", "component": "D", "qty_per": 3},
    {"parent": "D", "component": "E", "qty_per": 2},
])

procurement_type_map = {
    "A": "FNG",
    "B": "ASSEMBLY",
    "C": "RAW",
    "D": "ASSEMBLY",
    "E": "RAW"
}

lead_time_map = {"A": 2, "B": 1, "C": 7, "D": 1, "E":7}


lot_policy_map = {
    "A": "L4L",
    "B": "FIXED",
    "C": "COVER_DAYS",
    "D": "WEEKLY_CALENDAR",
    "E": "MIN_MAX"
}
## policy paramater include rounding_value(multiple), MOQ(min_lot_size), cover_days, max_level, week_day
    # ----------- {"rounding_value": 200, "MOQ": 200, "cover_days": 7, "up_to_level_qty": 1000}  
lot_policy_params = {
    "A": {},
    "B": {"rounding_value": 500, "MOQ": 500},
    "C": {"rounding_value": 1000, "MOQ": 1000, "cover_days": 7},
    "D": {"rounding_value": 5000, "MOQ": 5000, "week_day": "Tuesday"},
    "E": {"rounding_value": 5000, "MOQ": 5000, "max_level": 30000},
}


on_hand0 = {"A": 100, "B": 49, "C": 0, "D": 0, "E":15000}
safety_stock = {"A": 200, "B": 300, "C": 199, "D": 1000}


demand_records = [
    {"item": "A", "date": pd.Timestamp("2025-10-01"), "qty": 488},
    {"item": "A", "date": pd.Timestamp("2025-10-01"), "qty": 400},
    {"item": "A", "date": pd.Timestamp("2025-10-03"), "qty": 30000},
    {"item": "A", "date": pd.Timestamp("2025-10-04"), "qty": 100},
    {"item": "A", "date": pd.Timestamp("2025-10-05"), "qty": 19999},
    {"item": "A", "date": pd.Timestamp("2025-10-06"), "qty": 388},
    {"item": "A", "date": pd.Timestamp("2025-10-10"), "qty": 5000},
    {"item": "A", "date": pd.Timestamp("2025-10-11"), "qty": 6000},
    {"item": "A", "date": pd.Timestamp("2025-10-12"), "qty": 7000},
    {"item": "A", "date": pd.Timestamp("2025-10-13"), "qty": 1555},
    {"item": "A", "date": pd.Timestamp("2025-10-14"), "qty": 29999},
    {"item": "A", "date": pd.Timestamp("2025-10-15"), "qty": 799},
    {"item": "A", "date": pd.Timestamp("2025-10-16"), "qty": 1000},
    {"item": "A", "date": pd.Timestamp("2025-10-17"), "qty": 500},
    {"item": "A", "date": pd.Timestamp("2025-10-18"), "qty": 30000},
    {"item": "A", "date": pd.Timestamp("2025-10-19"), "qty": 1000},
    #------------------------
    {"item": "B", "date": pd.Timestamp("2025-10-01"), "qty": 488},
    {"item": "B", "date": pd.Timestamp("2025-10-01"), "qty": 400},
    {"item": "B", "date": pd.Timestamp("2025-10-03"), "qty": 30000},
    {"item": "B", "date": pd.Timestamp("2025-10-04"), "qty": 100},
    {"item": "B", "date": pd.Timestamp("2025-10-05"), "qty": 19999},
    {"item": "B", "date": pd.Timestamp("2025-10-06"), "qty": 388},
    {"item": "B", "date": pd.Timestamp("2025-10-10"), "qty": 5000},
    {"item": "B", "date": pd.Timestamp("2025-10-11"), "qty": 6000},
    {"item": "B", "date": pd.Timestamp("2025-10-12"), "qty": 7000},
    {"item": "B", "date": pd.Timestamp("2025-10-13"), "qty": 1555},
    {"item": "B", "date": pd.Timestamp("2025-10-14"), "qty": 29999},
    {"item": "B", "date": pd.Timestamp("2025-10-15"), "qty": 799},
    {"item": "B", "date": pd.Timestamp("2025-10-16"), "qty": 1000},
    {"item": "B", "date": pd.Timestamp("2025-10-17"), "qty": 500},
    {"item": "B", "date": pd.Timestamp("2025-10-18"), "qty": 30000},
    {"item": "B", "date": pd.Timestamp("2025-10-19"), "qty": 1000},
    
    {"item": "C", "date": pd.Timestamp("2025-10-01"), "qty": 488},
    {"item": "C", "date": pd.Timestamp("2025-10-01"), "qty": 400},
    {"item": "C", "date": pd.Timestamp("2025-10-03"), "qty": 30000},
    {"item": "C", "date": pd.Timestamp("2025-10-04"), "qty": 100},
    {"item": "C", "date": pd.Timestamp("2025-10-05"), "qty": 19999},
    {"item": "C", "date": pd.Timestamp("2025-10-06"), "qty": 388},
    {"item": "C", "date": pd.Timestamp("2025-10-10"), "qty": 5000},
    {"item": "C", "date": pd.Timestamp("2025-10-11"), "qty": 6000},
    {"item": "C", "date": pd.Timestamp("2025-10-12"), "qty": 7000},
    {"item": "C", "date": pd.Timestamp("2025-10-13"), "qty": 1555},
    {"item": "C", "date": pd.Timestamp("2025-10-14"), "qty": 29999},
    {"item": "C", "date": pd.Timestamp("2025-10-15"), "qty": 799},
    {"item": "C", "date": pd.Timestamp("2025-10-16"), "qty": 1000},
    {"item": "C", "date": pd.Timestamp("2025-10-17"), "qty": 500},
    {"item": "C", "date": pd.Timestamp("2025-10-18"), "qty": 30000},
    {"item": "C", "date": pd.Timestamp("2025-10-19"), "qty": 1000},
    
    {"item": "D", "date": pd.Timestamp("2025-10-01"), "qty": 488},
    {"item": "D", "date": pd.Timestamp("2025-10-01"), "qty": 400},
    {"item": "D", "date": pd.Timestamp("2025-10-03"), "qty": 30000},
    {"item": "D", "date": pd.Timestamp("2025-10-04"), "qty": 100},
    {"item": "D", "date": pd.Timestamp("2025-10-05"), "qty": 19999},
    {"item": "D", "date": pd.Timestamp("2025-10-06"), "qty": 388},
    {"item": "D", "date": pd.Timestamp("2025-10-10"), "qty": 5000},
    {"item": "D", "date": pd.Timestamp("2025-10-11"), "qty": 10000},
    {"item": "D", "date": pd.Timestamp("2025-10-12"), "qty": 5000},
    {"item": "D", "date": pd.Timestamp("2025-10-13"), "qty": 1555},
    {"item": "D", "date": pd.Timestamp("2025-10-14"), "qty": 19999},
    {"item": "D", "date": pd.Timestamp("2025-10-15"), "qty": 1300},
    {"item": "D", "date": pd.Timestamp("2025-10-16"), "qty": 1000},
    {"item": "D", "date": pd.Timestamp("2025-10-17"), "qty": 500},
    {"item": "D", "date": pd.Timestamp("2025-10-18"), "qty": 30000},
    {"item": "D", "date": pd.Timestamp("2025-10-19"), "qty": 1000},
      
      
    {"item": "E", "date": pd.Timestamp("2025-10-01"), "qty": 488},
    {"item": "E", "date": pd.Timestamp("2025-10-01"), "qty": 400},
    {"item": "E", "date": pd.Timestamp("2025-10-03"), "qty": 30000},
    {"item": "E", "date": pd.Timestamp("2025-10-04"), "qty": 100},
    {"item": "E", "date": pd.Timestamp("2025-10-05"), "qty": 19999},
    {"item": "E", "date": pd.Timestamp("2025-10-06"), "qty": 388},
    {"item": "E", "date": pd.Timestamp("2025-10-10"), "qty": 5000},
    {"item": "E", "date": pd.Timestamp("2025-10-11"), "qty": 10000},
    {"item": "E", "date": pd.Timestamp("2025-10-12"), "qty": 5000},
    {"item": "E", "date": pd.Timestamp("2025-10-13"), "qty": 1555},
    {"item": "E", "date": pd.Timestamp("2025-10-14"), "qty": 19999},
    {"item": "E", "date": pd.Timestamp("2025-10-15"), "qty": 1300},
    {"item": "E", "date": pd.Timestamp("2025-10-16"), "qty": 1000},
    {"item": "E", "date": pd.Timestamp("2025-10-17"), "qty": 500},
    {"item": "E", "date": pd.Timestamp("2025-10-18"), "qty": 30000},
    {"item": "E", "date": pd.Timestamp("2025-10-19"), "qty": 1000},
]

scheduled_receipts = pd.DataFrame([
    {"item": "A", "date": pd.Timestamp("2025-10-01"), "qty": 399}, 
    {"item": "A", "date": pd.Timestamp("2025-10-01"), "qty": 100},  
    {"item": "A", "date": pd.Timestamp("2025-10-05"), "qty": 199},  
    {"item": "A", "date": pd.Timestamp("2025-10-05"), "qty": 100},  
    {"item": "A", "date": pd.Timestamp("2025-10-12"), "qty": 15000},  
    #----------------------------------
    {"item": "B", "date": pd.Timestamp("2025-10-05"), "qty": 499},
    {"item": "C", "date": pd.Timestamp("2025-10-06"), "qty": 299},
    {"item": "C", "date": pd.Timestamp("2025-10-03"), "qty": 1000},
    {"item": "C", "date": pd.Timestamp("2025-10-03"), "qty": 555},
    #----------------------------------
    {"item": "D", "date": pd.Timestamp("2025-10-01"), "qty": 5000},
    {"item": "D", "date": pd.Timestamp("2025-10-03"), "qty": 25000},
    {"item": "D", "date": pd.Timestamp("2025-10-06"), "qty": 5000},
    {"item": "D", "date": pd.Timestamp("2025-10-08"), "qty": 5000},
    {"item": "D", "date": pd.Timestamp("2025-10-11"), "qty": 5000},
    {"item": "D", "date": pd.Timestamp("2025-10-15"), "qty": 5000},
    #----------------------------------
    {"item": "E", "date": pd.Timestamp("2025-10-01"), "qty": 5000},
    {"item": "E", "date": pd.Timestamp("2025-10-03"), "qty": 25000},
    {"item": "E", "date": pd.Timestamp("2025-10-06"), "qty": 5000},
    {"item": "E", "date": pd.Timestamp("2025-10-08"), "qty": 5000},
    {"item": "E", "date": pd.Timestamp("2025-10-11"), "qty": 5000},
    {"item": "E", "date": pd.Timestamp("2025-10-15"), "qty": 5000},
    
])


policy_master_records = []
for item, policy_p in lot_policy_params.items():
    policy_master_records.append({
        "item_code": item,
        "procurement_type" : procurement_type_map.get(item, "RAW"),
        "policy_name": lot_policy_map.get(item, "L4L"),
        "lead_time": lead_time_map.get(item, 1),
        "safety_stock": safety_stock.get(item, 0),
        "rounding_value": policy_p.get("rounding_value",1),
        "MOQ": policy_p.get("MOQ",1),
        "cover_days": policy_p.get("cover_days", ),
        "week_day": policy_p.get("week_day", ),
        "max_level": policy_p.get("max_level", ),

    })





In [23]:
"""FILE
---------------------
"""

TODAY = pd.Timestamp("2025-10-01")

# Master

item_master_df = pd.DataFrame([
    {"item_code": "A", "desc": "Finished Good A", "uom": "PCS", "vendor": "V001", "category": "FG"},
    {"item_code": "B", "desc": "Finished Good B", "uom": "PCS", "vendor": "V002", "category": "FG"},
    {"item_code": "C", "desc": "Component C", "uom": "PCS", "vendor": "V004", "category": "RM"},
    {"item_code": "E", "desc": "Finished Good D", "uom": "PCS", "vendor": "V004", "category": "FG"},
    {"item_code": "D", "desc": "Component E", "uom": "PCS", "vendor": "V005", "category": "RM"},
])
policy_master_df = pd.DataFrame(policy_master_records)
bom_master_df = bom_df.copy()

# Transaction table

demand_orders_df = pd.DataFrame(demand_records)
supply_orders_df = scheduled_receipts.copy()
onhand_df = pd.DataFrame([
    {"item_code": k, "date": TODAY, "qty": v } for k, v in on_hand0.items()  
])





In [24]:
# ---------------------------------------------------------------------
# Export all to CSV
# ---------------------------------------------------------------------

from pathlib import Path

path_master = r"input_fake/master"
path_transaction = r"input_fake/transaction"

Path(path_master).mkdir(parents=True, exist_ok=True)
Path(path_transaction).mkdir(parents=True, exist_ok=True)


item_master_df.to_csv("input_fake/master/item_master.csv", index=False)
policy_master_df.to_csv("input_fake/master/policy_master.csv", index=False)
bom_master_df.to_csv("input_fake/master/bom_master.csv", index=False)

demand_orders_df.to_csv("input_fake/transaction/demand_orders.csv", index=False)
supply_orders_df.to_csv("input_fake/transaction/supply_orders.csv", index=False)
onhand_df.to_csv("input_fake/transaction/onhand.csv", index=False)

print("✅ All MRP input CSVs generated successfully!")

✅ All MRP input CSVs generated successfully!
